In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")

In [4]:
import joblib
import pandas as pd
from sklearn.metrics import silhouette_score

features = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI',
            'DiabetesPedigreeFunction', 'Age']

df = pd.read_csv("../data/processed/diabetes_clustered.csv")
kmeans_final = joblib.load("../models/kmeans_model.joblib")
scaler_final = joblib.load("../models/cluster_scaler.joblib")

X_scaled = scaler_final.transform(df[features])
labels = kmeans_final.predict(X_scaled)
sil = silhouette_score(X_scaled, labels)

print(kmeans_final.n_clusters, round(kmeans_final.inertia_), round(sil, 3))

2 4071 0.232


c:\Users\yassi\Desktop\diabetes-risk-mlops\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but KMeans was fitted with feature names
  warnings.warn(


In [7]:

import mlflow


mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("diabetes-clustering")


with mlflow.start_run(run_name="kmeans-k2"):
    mlflow.log_param("k",kmeans_final.n_clusters)
    mlflow.log_param("features",", ".join(features))
    
    for name,mean,std in zip(features, scaler_final.mean_, scaler_final.scale_):
        mlflow.log_param(f"scaler_mean_{name}",round(mean,4))
        mlflow.log_param(f"scaler_std_{name}",round(std,4))
        
        
    mlflow.log_metric("inertia",kmeans_final.inertia_)
    mlflow.log_metric("silhouette_score", sil)
    
    mlflow.log_artifact("../models/kmeans_model.joblib")
    mlflow.log_artifact("../models/cluster_scaler.joblib")

print("Clustering run saved")

Clustering run saved


In [17]:
from src.tracking import log_clustering_run

log_clustering_run(
    kmeans_final, scaler_final, features, sil,
    model_path="../models/kmeans_model.joblib",
    scaler_path="../models/cluster_scaler.joblib",
)

In [18]:
from sklearn.model_selection import train_test_split
from src.tracking import log_classification_run

X = df[features]
y = df["Cluster"]
_, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

pipeline = joblib.load("../models/diabetes_risk_pipeline.joblib")
log_classification_run(
    pipeline, X_test, y_test,
    params={"model": "LogisticRegression", "C": 10, "solver": "lbfgs"},
    features=features,
    run_name="logistic-regression-tuned",
    model_path="../models/diabetes_risk_pipeline.joblib",
)

In [19]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

experiments = {
    "logistic-untuned": (LogisticRegression(random_state=42, max_iter=1000),{"model": "LogisticRegression", "tuned": "no"}),
    "logistic-tuned":   (LogisticRegression(C=10, random_state=42, max_iter=1000),{"model": "LogisticRegression", "tuned": "yes", "C": 10}),
    
    "rf-untuned":       (RandomForestClassifier(random_state=42),{"model": "RandomForest", "tuned": "no"}),
    "rf-tuned":         (RandomForestClassifier(max_depth=5, min_samples_split=5, n_estimators=100, random_state=42),{"model": "RandomForest", "tuned": "yes", "max_depth": 5, "min_samples_split": 5, "n_estimators": 100}),
    
    "gb-untuned":       (GradientBoostingClassifier(random_state=42),{"model": "GradientBoosting", "tuned": "no"}),
    "gb-tuned":         (GradientBoostingClassifier(learning_rate=0.1, max_depth=3, n_estimators=100, random_state=42),{"model": "GradientBoosting", "tuned": "yes", "learning_rate": 0.1, "max_depth": 3, "n_estimators": 100}),
}


for run_name, (model, params) in experiments.items():
    pipe = Pipeline([("scaler", StandardScaler()), ("model", model)])
    pipe.fit(X_train, y_train)

    path = f"../models/{run_name}.joblib"
    joblib.dump(pipe, path)

    log_classification_run(pipe, X_test, y_test, params=params, features=features,run_name=run_name, model_path=path)


In [20]:
import joblib
p = joblib.load("../models/diabetes_risk_pipeline.joblib")
print(p.named_steps["model"].C)

10


In [ ]:
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
from src.tracking import TRACKING_URI


pipe = joblib.load("../models/diabetes_risk_pipeline.joblib")

mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment("diabetes-classification")

with mlflow.start_run(run_name="final-logistic-pipeline"):
    mlflow.sklearn.log_model(
        pipe,
        name="model",
        registered_model_name="diabetes-risk-classifier",
    )

3.17.0


Successfully registered model 'diabetes-risk-classifier'.
Created version '1' of model 'diabetes-risk-classifier'.


In [24]:
from mlflow import MlflowClient
from src.tracking import TRACKING_URI

client = MlflowClient(tracking_uri=TRACKING_URI)

versions = client.search_model_versions("name='diabetes-risk-classifier'")
latest = max(int(v.version) for v in versions)
print("Versions found:", [v.version for v in versions])

client.set_registered_model_alias("diabetes-risk-classifier", "Production", latest)
print("Version", latest, "is now Production")

Versions found: [1]
Version 1 is now Production


In [25]:
mlflow.set_tracking_uri(TRACKING_URI)
model = mlflow.sklearn.load_model("models:/diabetes-risk-classifier@Production")
print(model.predict(X_test.head(3)))

[0 1 0]
